In [ ]:
import hashlib
# CELL 1 — FINAL TEST AUTHORIZATION
# The user explicitly requested the final locked TEST.
AUTHORIZE_LOCKED_TEST = True
LOCKED_TEST_ACK = "I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST"

PUBLICATION_SEEDS = [42, 1337, 2026, 31415, 271828]
EXPECTED_GATED_STEP = {42: 800, 1337: 932, 2026: 800, 31415: 932, 271828: 932}

MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
MODEL_REVISION = '246e42c643bd9653381dafe9cd5a2b2eb51944b7'
FACETGATE_COMMON_SHA256 = '735b3e08413949cdb148fe4ce570493b5cd410e562dad7a12748b2e5f2fd6a38'
PARENT_GATED_PROTOCOL_SHA256 = '627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd'
TASK_INSTRUCTION_SHA256 = 'bfef1e522a9433e1ca0c57a3a1bea49cd890d8428308b7ffdf6b24e4fe51667f'
QWEN_PARENT_INTERFACE_SHA256 = 'f5a12f339bc537baa4dcc83c24c466c5ba60943f34964a9999fb2a0bbf43ce04'

EXPECTED_LABEL_SHA256 = '9cb9afdd88c9a750d3eb6f36aa512dc6ad5b3067aedfac79b138b7fef64a9bb6'
EXPECTED_OCR_SHA256 = 'eed35bf77f749e98d9222fd571d149b99c4c6a610e29bc5db5783d3489492c0c'
EXPECTED_SPLIT_SHA256 = 'bbcf7e13fc0aab4ac3b5f58a2ebf8880740ed60ad7ecb2cb61910b0497d1b4f0'
EXPECTED_CANONICAL_ROW_SHA256 = '440f8e6982cf4272fc0b6c3e3e6fc9b079a02ce01cd5350a02be0d4a9fa1cf39'
EXPECTED_TEST_IDENTITY_SHA256 = '5f6ac881e9026d682f2f41d65ed1f08e1861d8c8f030bae04d98f9b9db6419db'
EXPECTED_IMAGE_BUNDLE_SHA256 = '9e134f26c63c23ba871c954ee1d7e03499a47ac3f9bd49d12e0c5ffa977665af'

TASK_INSTRUCTION = (
    "Encode this meme for six independent annotations: offensive, vulgar, abuse, "
    "sarcastic, language (Bangla/English/Banglish), and zero or more of seven real "
    "target groups. Do not impose logical implications between tasks. "
)
if hashlib.sha256(TASK_INSTRUCTION.encode("utf-8")).hexdigest() != TASK_INSTRUCTION_SHA256:
    raise RuntimeError("Task-instruction SHA drift")

if not AUTHORIZE_LOCKED_TEST or LOCKED_TEST_ACK != "I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST":
    raise RuntimeError("Final locked TEST is not explicitly authorized.")

print({
    "mode": "QWEN3_FACETGATE_5SEED_FINAL_LOCKED_TEST",
    "seeds": PUBLICATION_SEEDS,
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "authorization": True,
    "scientific_rule": "Gated Core5 unchanged; FacetGate replaces Target only",
})


In [ ]:
# CELL 2 — PINNED KAGGLE RUNTIME
# Scientific settings are unchanged. This only reproduces the validated runtime.
import importlib.metadata as md, subprocess, sys, os
from pathlib import Path

protected = ["torch","torchvision","numpy","pandas","scikit-learn","scipy","Pillow"]
before = {}
for pkg in protected:
    try:
        before[pkg] = md.version(pkg)
    except Exception:
        pass

constraints = Path("/kaggle/working/facetgate_final_test_constraints.txt")
constraints.write_text(
    "\n".join(f"{k}=={v}" for k,v in before.items()) + "\n",
    encoding="utf-8"
)

cmd = [
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "--upgrade-strategy", "only-if-needed", "-c", str(constraints),
    "transformers==5.15.0", "peft==0.20.0", "accelerate==1.14.0",
    "bitsandbytes==0.50.1", "safetensors==0.8.0"
]
print("[PIP]", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

after = {}
for pkg in protected:
    try:
        after[pkg] = md.version(pkg)
    except Exception:
        pass

if before != after:
    raise RuntimeError(f"Protected base-package drift: before={before} after={after}")

print("[RUNTIME PASS]", after)


In [ ]:
# CELL 3 — CPU-ONLY FACETGATE + GATED ASSET FREEZE
# This cell MUST finish for all five seeds before any TEST model forward is permitted.
import os, json, hashlib, zipfile, shutil, math, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/M3Offend_9318_Qwen3VL8B_FacetGate_5Seed_LOCKED_TEST_v1_0")
TEMP_ROOT = Path("/kaggle/temp/qwen3_facetgate_final_test")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

BINARY_TASKS = ["offensive","vulgar","abuse","sarcastic"]
LANGUAGE_LABELS = ["Bangla","English","Banglish"]
TARGET_LABELS = ["Individual","Gender","Political","Social","National","Religious","Others"]

def sha256_file(path, chunk=8<<20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def json_sha(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, separators=(",",":"), ensure_ascii=False).encode("utf-8")
    ).hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    os.replace(tmp, path)

def unwrap_state(obj):
    if isinstance(obj, dict) and isinstance(obj.get("discriminative"), dict):
        return obj["discriminative"]
    if isinstance(obj, dict) and isinstance(obj.get("state_dict"), dict):
        return obj["state_dict"]
    return obj

def inference_fingerprint(cp):
    cp = Path(cp)
    heads = cp / "classifier_heads.pt"
    acfg = cp / "adapter" / "adapter_config.json"
    aw = next(
        (cp / "adapter" / n for n in ["adapter_model.safetensors","adapter_model.bin"] if (cp/"adapter"/n).is_file()),
        None
    )
    if not heads.is_file() or not acfg.is_file() or aw is None:
        return None
    return hashlib.sha256(
        (sha256_file(heads) + "|" + sha256_file(aw) + "|" + sha256_file(acfg)).encode()
    ).hexdigest()

def checkpoint_inference_complete(cp):
    cp = Path(cp)
    return (
        (cp/"classifier_heads.pt").is_file()
        and (cp/"adapter"/"adapter_config.json").is_file()
        and any((cp/"adapter"/n).is_file() for n in ["adapter_model.safetensors","adapter_model.bin"])
    )

# ------------------------------------------------------------
# FacetGate validation-package discovery
# ------------------------------------------------------------
REQUIRED_FACET_FILES = [
    "BASE_GATED_BINDING.json",
    "FEATURE_CACHE_MANIFEST.json",
    "facetgate_feature_cache.npz",
    "candidate_metrics.jsonl",
    "selected_checkpoint.pt",
    "validation_predictions.csv",
    "validation_thresholds.json",
    "bootstrap_results.json",
    "facetgate_result.json",
    "parent_gated_thresholds.normalized.json",
    "dataset_audit.json",
    "run_manifest.json",
]

def validate_facet_root(root, seed):
    root = Path(root)
    missing = [n for n in REQUIRED_FACET_FILES if not (root/n).is_file()]
    if missing:
        return None, f"missing={missing}"

    try:
        rm = json.loads((root/"run_manifest.json").read_text(encoding="utf-8"))
        br = json.loads((root/"BASE_GATED_BINDING.json").read_text(encoding="utf-8"))
        fm = json.loads((root/"FEATURE_CACHE_MANIFEST.json").read_text(encoding="utf-8"))
        fr = json.loads((root/"facetgate_result.json").read_text(encoding="utf-8"))
        vt = json.loads((root/"validation_thresholds.json").read_text(encoding="utf-8"))
    except Exception as e:
        return None, f"json_read={type(e).__name__}"

    if rm.get("schema") != "M3Offend_FacetGate_v1_0_RunManifest":
        return None, "wrong_run_manifest_schema"
    if rm.get("protocol_sha256") != FACETGATE_COMMON_SHA256:
        return None, "facetgate_protocol_sha_mismatch"
    if rm.get("parent_gated_protocol_sha256") != PARENT_GATED_PROTOCOL_SHA256:
        return None, "parent_gated_protocol_sha_mismatch"
    if rm.get("model_id") != MODEL_ID or rm.get("model_revision") != MODEL_REVISION:
        return None, "model_identity_mismatch"
    if int(rm.get("seed",-1)) != int(seed):
        return None, "seed_mismatch"
    if int(rm.get("locked_test_model_forwards",-1)) != 0:
        return None, "development_package_touched_TEST"

    if br.get("protocol_sha256") != FACETGATE_COMMON_SHA256:
        return None, "BASE_GATED_BINDING_protocol_mismatch"
    if br.get("parent_gated_protocol_sha256") != PARENT_GATED_PROTOCOL_SHA256:
        return None, "BASE_GATED_BINDING_parent_protocol_mismatch"
    if br.get("model_id") != MODEL_ID or br.get("model_revision") != MODEL_REVISION:
        return None, "BASE_GATED_BINDING_model_mismatch"
    if int(br.get("seed",-1)) != int(seed):
        return None, "BASE_GATED_BINDING_seed_mismatch"
    if int(br.get("selected_gated_step",-1)) != int(EXPECTED_GATED_STEP[seed]):
        return None, f"wrong_gated_step={br.get('selected_gated_step')}"
    if int(br.get("test_model_forwards",-1)) != 0:
        return None, "BASE_GATED_BINDING_touched_TEST"

    if fm.get("schema") != "M3Offend_FacetGate_v1_0_FEATURE_CACHE_MANIFEST":
        return None, "feature_manifest_schema_mismatch"
    if fm.get("protocol_sha256") != FACETGATE_COMMON_SHA256:
        return None, "feature_manifest_protocol_mismatch"
    if fm.get("model_id") != MODEL_ID or fm.get("model_revision") != MODEL_REVISION:
        return None, "feature_manifest_model_mismatch"
    if int(fm.get("seed",-1)) != int(seed):
        return None, "feature_manifest_seed_mismatch"
    if fm.get("feature_binding_sha256") != QWEN_PARENT_INTERFACE_SHA256:
        return None, "Qwen_feature_binding_mismatch"
    if int(fm.get("rows",-1)) != 8386 or int(fm.get("train_rows",-1)) != 7454 or int(fm.get("validation_rows",-1)) != 932:
        return None, "development_feature_cache_row_drift"
    if int(fm.get("representation_dim",-1)) != 1024:
        return None, "representation_dim_drift"
    if int(fm.get("locked_test_model_forwards",-1)) != 0:
        return None, "feature_cache_touched_TEST"

    if fr.get("protocol_sha256") != FACETGATE_COMMON_SHA256:
        return None, "facetgate_result_protocol_mismatch"
    if fr.get("model_id") != MODEL_ID or fr.get("model_revision") != MODEL_REVISION:
        return None, "facetgate_result_model_mismatch"
    if int(fr.get("seed",-1)) != int(seed):
        return None, "facetgate_result_seed_mismatch"
    if int(fr.get("locked_test_model_forwards",-1)) != 0:
        return None, "facetgate_result_touched_TEST"

    if vt.get("source_split") != "validation" or int(vt.get("source_rows",-1)) != 932:
        return None, "FacetGate_threshold_source_mismatch"
    if not isinstance(vt.get("thresholds"), dict) or set(vt["thresholds"]) != set(TARGET_LABELS):
        return None, "FacetGate_threshold_schema_mismatch"

    # Verify all artifacts explicitly bound by run_manifest.
    for name, expected in rm.get("artifacts", {}).items():
        p = root / name
        if not p.is_file():
            return None, f"manifest_artifact_missing={name}"
        if sha256_file(p) != expected:
            return None, f"manifest_artifact_SHA_mismatch={name}"

    if sha256_file(root/"facetgate_feature_cache.npz") != fm.get("feature_cache_sha256"):
        return None, "feature_cache_SHA_mismatch"

    state = torch.load(root/"selected_checkpoint.pt", map_location="cpu", weights_only=False)
    if int(state.get("step",-1)) != int(fr.get("selected_facetgate_step",-2)):
        return None, "selected_FacetGate_step_mismatch"

    return {
        "root": str(root),
        "run_manifest": rm,
        "base_binding": br,
        "feature_manifest": fm,
        "facet_result": fr,
        "facet_thresholds": vt,
        "selected_facet_step": int(fr["selected_facetgate_step"]),
        "selected_facet_sha256": sha256_file(root/"selected_checkpoint.pt"),
        "facet_threshold_sha256": sha256_file(root/"validation_thresholds.json"),
        "binding_sha256": rm.get("binding_sha256"),
    }, "ok"

def direct_facet_candidates(seed):
    hits = []
    for rm in INPUT_ROOT.rglob("run_manifest.json"):
        try:
            obj = json.loads(rm.read_text(encoding="utf-8"))
            if (
                obj.get("schema") == "M3Offend_FacetGate_v1_0_RunManifest"
                and obj.get("protocol_sha256") == FACETGATE_COMMON_SHA256
                and obj.get("model_id") == MODEL_ID
                and obj.get("model_revision") == MODEL_REVISION
                and int(obj.get("seed",-1)) == int(seed)
            ):
                rec, why = validate_facet_root(rm.parent, seed)
                if rec is not None:
                    hits.append(rec)
        except Exception:
            pass
    return hits

def extract_facet_zip_candidates(seed):
    hits = []
    for zp in INPUT_ROOT.rglob("*.zip"):
        try:
            if zp.stat().st_size > 8_000_000_000:
                continue
            with zipfile.ZipFile(zp) as zf:
                names = set(zf.namelist())
                rms = [n for n in names if n.endswith("run_manifest.json")]
                for rm_member in rms:
                    try:
                        obj = json.loads(zf.read(rm_member).decode("utf-8"))
                        if not (
                            obj.get("schema") == "M3Offend_FacetGate_v1_0_RunManifest"
                            and obj.get("protocol_sha256") == FACETGATE_COMMON_SHA256
                            and obj.get("model_id") == MODEL_ID
                            and obj.get("model_revision") == MODEL_REVISION
                            and int(obj.get("seed",-1)) == int(seed)
                        ):
                            continue
                        prefix = rm_member[:-len("run_manifest.json")]
                        needed_members = {}
                        for name in REQUIRED_FACET_FILES:
                            member = prefix + name
                            if member not in names:
                                break
                            needed_members[name] = member
                        if len(needed_members) != len(REQUIRED_FACET_FILES):
                            continue
                        dest = TEMP_ROOT / f"facet_seed{seed}_{hashlib.sha256((str(zp)+prefix).encode()).hexdigest()[:12]}"
                        if dest.exists():
                            shutil.rmtree(dest)
                        dest.mkdir(parents=True)
                        for name, member in needed_members.items():
                            (dest/name).parent.mkdir(parents=True, exist_ok=True)
                            (dest/name).write_bytes(zf.read(member))
                        rec, why = validate_facet_root(dest, seed)
                        if rec is not None:
                            rec["zip_source"] = str(zp)
                            hits.append(rec)
                    except Exception:
                        continue
        except Exception:
            continue
    return hits

FACET = {}
for seed in PUBLICATION_SEEDS:
    hits = direct_facet_candidates(seed)
    if not hits:
        hits = extract_facet_zip_candidates(seed)
    if not hits:
        raise RuntimeError(
            f"No complete frozen FacetGate validation package found for seed={seed}. "
            "Add that completed FacetGate notebook OUTPUT as a Kaggle input."
        )
    identities = {
        (h["selected_facet_sha256"], h["facet_threshold_sha256"], h["binding_sha256"])
        for h in hits
    }
    if len(identities) != 1:
        raise RuntimeError(f"Conflicting FacetGate packages for seed={seed}: {hits}")
    FACET[seed] = sorted(hits, key=lambda h: len(h["root"]))[0]

print("[FACETGATE 5-SEED PACKAGE DISCOVERY PASS]")
for seed in PUBLICATION_SEEDS:
    r = FACET[seed]
    print(seed, {
        "root": r["root"],
        "Gated_step": r["base_binding"]["selected_gated_step"],
        "FacetGate_step": r["selected_facet_step"],
        "FacetGate_checkpoint_sha256": r["selected_facet_sha256"],
        "validation_threshold_sha256": r["facet_threshold_sha256"],
        "TEST_forwards": 0,
    })

# ------------------------------------------------------------
# Gated inference-state discovery.
# The FacetGate run binds an inference fingerprint:
# SHA256(head_sha | adapter_sha | adapter_config_sha).
# ------------------------------------------------------------
def manifest_seed_ok(cp, seed):
    mf = Path(cp) / "checkpoint_manifest.json"
    if not mf.is_file():
        return False
    try:
        obj = json.loads(mf.read_text(encoding="utf-8"))
    except Exception:
        return False
    vals = []
    def walk(x):
        if isinstance(x,dict):
            yield x
            for v in x.values():
                yield from walk(v)
        elif isinstance(x,(list,tuple)):
            for v in x:
                yield from walk(v)
    for d in walk(obj):
        for k,v in d.items():
            if str(k).casefold() in {"seed","run_seed","training_seed","model_seed"} and isinstance(v,(int,float)) and float(v).is_integer():
                vals.append(int(v))
    return sorted(set(vals)) == [int(seed)]

def direct_gated_candidates(seed, expected_fp, step):
    out = []
    for nm in [f"checkpoint-{step}", f"checkpoint-{step:04d}"]:
        for cp in INPUT_ROOT.rglob(nm):
            try:
                if not cp.is_dir() or not checkpoint_inference_complete(cp):
                    continue
                if not manifest_seed_ok(cp, seed):
                    continue
                fp = inference_fingerprint(cp)
                if fp == expected_fp:
                    out.append({"kind":"directory","path":str(cp),"fingerprint":fp})
            except Exception:
                pass
    return out

def zip_gated_candidates(seed, expected_fp, step):
    out = []
    for zp in INPUT_ROOT.rglob("*.zip"):
        try:
            if zp.stat().st_size > 8_000_000_000:
                continue
            with zipfile.ZipFile(zp) as zf:
                names = set(zf.namelist())
                heads_members = [
                    n for n in names
                    if n.endswith(f"checkpoint-{step}/classifier_heads.pt")
                    or n.endswith(f"checkpoint-{step:04d}/classifier_heads.pt")
                ]
                for hm in heads_members:
                    prefix = hm[:-len("classifier_heads.pt")]
                    acfg = prefix + "adapter/adapter_config.json"
                    aw = next(
                        (prefix + "adapter/" + n for n in ["adapter_model.safetensors","adapter_model.bin"]
                         if prefix + "adapter/" + n in names),
                        None
                    )
                    mf = prefix + "checkpoint_manifest.json"
                    if acfg not in names or aw is None or mf not in names:
                        continue
                    try:
                        mobj = json.loads(zf.read(mf).decode("utf-8"))
                    except Exception:
                        continue
                    vals = []
                    def walk(x):
                        if isinstance(x,dict):
                            yield x
                            for v in x.values(): yield from walk(v)
                        elif isinstance(x,(list,tuple)):
                            for v in x: yield from walk(v)
                    for d in walk(mobj):
                        for k,v in d.items():
                            if str(k).casefold() in {"seed","run_seed","training_seed","model_seed"} and isinstance(v,(int,float)) and float(v).is_integer():
                                vals.append(int(v))
                    if sorted(set(vals)) != [int(seed)]:
                        continue
                    head_sha = hashlib.sha256(zf.read(hm)).hexdigest()
                    cfg_sha = hashlib.sha256(zf.read(acfg)).hexdigest()
                    aw_sha = hashlib.sha256(zf.read(aw)).hexdigest()
                    fp = hashlib.sha256((head_sha+"|"+aw_sha+"|"+cfg_sha).encode()).hexdigest()
                    if fp != expected_fp:
                        continue
                    dest = TEMP_ROOT / f"gated_seed{seed}_{expected_fp[:12]}"
                    if dest.exists():
                        shutil.rmtree(dest)
                    dest.mkdir(parents=True)
                    for member in [hm, acfg, aw, mf]:
                        rel = member[len(prefix):]
                        p = dest / rel
                        p.parent.mkdir(parents=True, exist_ok=True)
                        p.write_bytes(zf.read(member))
                    if inference_fingerprint(dest) != expected_fp:
                        raise RuntimeError(f"Extracted Gated inference fingerprint drift seed={seed}")
                    out.append({"kind":"zip","path":str(dest),"source_zip":str(zp),"fingerprint":fp})
        except RuntimeError:
            raise
        except Exception:
            continue
    return out

GATED = {}
missing_gated = []
for seed in PUBLICATION_SEEDS:
    br = FACET[seed]["base_binding"]
    expected_fp = br.get("checkpoint_fingerprint")
    step = int(br["selected_gated_step"])
    if not expected_fp:
        raise RuntimeError(f"FacetGate BASE_GATED_BINDING lacks checkpoint_fingerprint seed={seed}")
    hits = direct_gated_candidates(seed, expected_fp, step)
    if not hits:
        hits = zip_gated_candidates(seed, expected_fp, step)
    if not hits:
        missing_gated.append({
            "seed":seed,
            "step":step,
            "expected_inference_fingerprint":expected_fp,
            "original_parent_path_recorded_by_FacetGate":br.get("checkpoint_dir"),
        })
        continue
    paths = sorted({h["path"] for h in hits})
    # All already match one exact expected inference fingerprint; mirrors are safe.
    GATED[seed] = sorted(hits, key=lambda h:(0 if h["kind"]=="directory" else 1, len(h["path"])))[0]

if missing_gated:
    print("\n[MISSING GATED INFERENCE STATES]")
    print(json.dumps(missing_gated, indent=2))
    raise RuntimeError(
        "The five FacetGate outputs prove which Gated parents were used, but the large Gated "
        "adapter/head weights are not guaranteed to be packaged inside those FacetGate outputs. "
        "Add the ORIGINAL Gated notebook outputs (or their Resume_Safeguard ZIPs) for the seeds "
        "listed above. TEST has NOT been touched."
    )

print("[GATED 5-SEED INFERENCE-STATE DISCOVERY PASS]")
for seed in PUBLICATION_SEEDS:
    print(seed, GATED[seed])

ASSET_FREEZE = {
    "facetgate_protocol_sha256": FACETGATE_COMMON_SHA256,
    "gated_protocol_sha256": PARENT_GATED_PROTOCOL_SHA256,
    "qwen_feature_binding_sha256": QWEN_PARENT_INTERFACE_SHA256,
    "seeds": {},
    "test_model_forwards": 0,
}
for seed in PUBLICATION_SEEDS:
    ASSET_FREEZE["seeds"][str(seed)] = {
        "gated_step": int(FACET[seed]["base_binding"]["selected_gated_step"]),
        "gated_inference_fingerprint": FACET[seed]["base_binding"]["checkpoint_fingerprint"],
        "facetgate_step": FACET[seed]["selected_facet_step"],
        "facetgate_checkpoint_sha256": FACET[seed]["selected_facet_sha256"],
        "facetgate_validation_threshold_sha256": FACET[seed]["facet_threshold_sha256"],
        "facetgate_binding_sha256": FACET[seed]["binding_sha256"],
    }
atomic_json(WORK_ROOT/"FINAL_PRETEST_ASSET_FREEZE.json", ASSET_FREEZE)
print("[PRETEST ASSET FREEZE WRITTEN]", WORK_ROOT/"FINAL_PRETEST_ASSET_FREEZE.json")


In [ ]:
# CELL 4 — CANONICAL 9,318 DATASET + ORDERED TEST AUDIT
# No model forward occurs here.
import ast, re, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

def read_table(path):
    p = Path(path)
    if p.suffix.lower()==".csv":
        return pd.read_csv(p)
    if p.suffix.lower() in {".jsonl",".ndjson"}:
        return pd.read_json(p, lines=True)
    obj = json.loads(p.read_text(encoding="utf-8"))
    if isinstance(obj,list):
        return pd.DataFrame(obj)
    if isinstance(obj,dict):
        for k in ["data","records","items","annotations","samples"]:
            if isinstance(obj.get(k),list):
                return pd.DataFrame(obj[k])
        if obj and all(isinstance(v,dict) for v in obj.values()):
            return pd.DataFrame([{"__dict_key__":k,**v} for k,v in obj.items()])
    raise ValueError(f"Unsupported table structure: {p}")

def resolve_col(df,cands,what,required=True):
    low={str(c).casefold():c for c in df.columns}
    for c in cands:
        if c.casefold() in low:
            return low[c.casefold()]
    if required:
        raise KeyError(f"Cannot resolve {what}; columns={list(df.columns)}")
    return None

def image_name(v):
    return str(v).replace("\\","/").split("/")[-1]

def norm_key(v):
    s=image_name(v).strip().casefold()
    return re.sub(r"\.(jpg|jpeg|png|webp|bmp)$","",s)

def norm_binary(v,task):
    if isinstance(v,(bool,np.bool_)):
        return "Yes" if bool(v) else "No"
    s=str(v).strip().casefold()
    if s in {"yes","y","1","true","positive","pos"}:
        return "Yes"
    if s in {"no","n","0","false","negative","neg"}:
        return "No"
    raise ValueError(f"{task}: bad binary label {v!r}")

def norm_language(v):
    s=" ".join(str(v).strip().split()).casefold()
    aliases={
        "bangla":"Bangla","bengali":"Bangla","english":"English",
        "banglish":"Banglish","banglish (bangla + english)":"Banglish",
        "banglish(bangla + english)":"Banglish","bangla + english":"Banglish",
        "bangla+english":"Banglish","bengali + english":"Banglish",
        "romanized bangla":"Banglish","romanised bangla":"Banglish",
    }
    if s not in aliases:
        raise ValueError(f"bad language {v!r}")
    return aliases[s]

def norm_target(v):
    if v is None or (isinstance(v,float) and np.isnan(v)):
        vals=[]
    elif isinstance(v,(list,tuple,set)):
        vals=list(v)
    elif isinstance(v,str):
        s=v.strip()
        if not s:
            vals=[]
        else:
            try:
                q=json.loads(s); vals=q if isinstance(q,list) else [q]
            except Exception:
                try:
                    q=ast.literal_eval(s); vals=list(q) if isinstance(q,(list,tuple,set)) else [q]
                except Exception:
                    vals=re.split(r"[,;|]",s)
    else:
        vals=[v]
    canon={x.casefold():x for x in TARGET_LABELS}
    canon["other"]="Others"
    out=[]
    for x in vals:
        sx=str(x).strip()
        if not sx or sx.casefold() in {"none","no target","null","nan"}:
            continue
        if sx.casefold() not in canon:
            raise ValueError(f"bad target {x!r}")
        c=canon[sx.casefold()]
        if c not in out:
            out.append(c)
    return [x for x in TARGET_LABELS if x in out]

def parse_labelstudio(df):
    rows=[]; errors=[]
    for rn,r in df.iterrows():
        try:
            data=r.get("data",{})
            if isinstance(data,str):
                data=json.loads(data)
            image_value=None
            if isinstance(data,dict):
                for k in ["image","image_id","image_name","filename","file_name","source_image_name"]:
                    if data.get(k) not in [None,""]:
                        image_value=data[k]; break
            if image_value is None:
                for k in ["image","image_id","image_name","filename","file_name","source_image_name","id"]:
                    if k in r and pd.notna(r[k]):
                        image_value=r[k]; break
            anns=r.get("annotations",[])
            if isinstance(anns,str):
                anns=json.loads(anns)
            if not isinstance(anns,list) or not anns:
                raise ValueError("no annotations")
            coll={k:[] for k in BINARY_TASKS+["language","target"]}
            for ann in anns:
                for res in (ann.get("result",[]) if isinstance(ann,dict) else []):
                    if not isinstance(res,dict):
                        continue
                    frm=str(res.get("from_name","")).casefold()
                    val=res.get("value",{}) or {}
                    choices=val.get("choices",val.get("labels",[]))
                    if isinstance(choices,str):
                        choices=[choices]
                    task=next((k for k in coll if k in frm),None)
                    if task and choices:
                        coll[task].append(list(choices))
            item={"record_key":norm_key(image_value),"label_source_id":str(image_value)}
            for t in BINARY_TASKS:
                vals=[norm_binary(b[0],t) for b in coll[t] if len(b)==1]
                if not vals or len(set(vals))!=1:
                    raise ValueError(f"{t} non-unanimous/missing {vals}")
                item[t]=vals[0]
            lv=[norm_language(b[0]) for b in coll["language"] if len(b)==1]
            if not lv or len(set(lv))!=1:
                raise ValueError(f"language non-unanimous/missing {lv}")
            item["language"]=lv[0]
            tv=[tuple(norm_target(b)) for b in coll["target"] if b]
            if not tv:
                tv=[tuple()]
            if len(set(tv))!=1:
                raise ValueError(f"target non-unanimous {tv}")
            item["target_list"]=list(tv[0])
            rows.append(item)
        except Exception as e:
            errors.append({"row":int(rn),"error":repr(e)})
    if errors:
        raise RuntimeError(f"Label parse errors={len(errors)} first={errors[:3]}")
    return pd.DataFrame(rows)

def parse_labels(path):
    df=read_table(path).copy()
    if "annotations" in df.columns:
        out=parse_labelstudio(df)
    else:
        idc=resolve_col(df,["image_id","image_name","source_image_name","source_image","image","filename","file_name","path","name","id","__dict_key__"],"label id")
        out=pd.DataFrame({"record_key":df[idc].map(norm_key),"label_source_id":df[idc].astype(str)})
        for t in BINARY_TASKS:
            c=resolve_col(df,[t],t)
            out[t]=df[c].map(lambda x:norm_binary(x,t))
        lc=resolve_col(df,["language","lang"],"language")
        out["language"]=df[lc].map(norm_language)
        tc=resolve_col(df,["target","targets"],"target")
        out["target_list"]=df[tc].map(norm_target)
    if out.record_key.duplicated().any():
        raise RuntimeError("duplicate label IDs")
    return out

def parse_ocr(path):
    df=read_table(path).copy()
    idc=resolve_col(df,["image_id","image_name","source_image_name","source_image","image","filename","file_name","path","name","id","__dict_key__"],"OCR id")
    tc=resolve_col(df,["cleaned_ocr","ocr_cleaned","ocr","text","ocr_text","clean_text"],"OCR text")
    out=pd.DataFrame({"record_key":df[idc].map(norm_key),"cleaned_ocr":df[tc].fillna("").astype(str)})
    if out.record_key.duplicated().any():
        raise RuntimeError("duplicate OCR IDs")
    return out

def parse_manifest(path):
    df=read_table(path).copy()
    idc=resolve_col(df,["image_id","image","image_name","filename","file_name","source_image_name"],"manifest id")
    sc=resolve_col(df,["split","set","partition","dataset_split"],"split")
    ic=resolve_col(df,["official_row_index","row_index","index"],"row index",False)
    out=pd.DataFrame({
        "record_key":df[idc].map(norm_key),
        "image_id":df[idc].map(image_name),
        "split":df[sc].astype(str).str.strip().str.casefold().replace({"val":"validation","valid":"validation","dev":"validation"}),
        "official_row_index":np.arange(len(df),dtype=int) if ic is None else pd.to_numeric(df[ic],errors="raise").astype(int)
    })
    if out.record_key.duplicated().any() or out.official_row_index.duplicated().any():
        raise RuntimeError("manifest IDs/indices not unique")
    return out.sort_values("official_row_index",kind="stable").reset_index(drop=True)

# Discover exact metadata by immutable SHA.
metadata=[]
for p in INPUT_ROOT.rglob("*"):
    try:
        if p.is_file() and p.suffix.lower() in {".json",".jsonl",".ndjson",".csv"} and p.stat().st_size<250_000_000:
            metadata.append(p)
    except OSError:
        pass

wanted={
    "labels":EXPECTED_LABEL_SHA256,
    "ocr":EXPECTED_OCR_SHA256,
    "split":EXPECTED_SPLIT_SHA256,
}
found={}
for i,p in enumerate(metadata,1):
    h=sha256_file(p)
    for role,expected in wanted.items():
        if h==expected:
            found[role]=p
    if i%100==0:
        print(f"[META SHA] {i}/{len(metadata)}",flush=True)

if set(found)!=set(wanted):
    raise FileNotFoundError(
        f"Canonical metadata not found by SHA. Found={found}. "
        "Add the canonical M3Offend dataset as an input."
    )

labels=parse_labels(found["labels"])
ocr=parse_ocr(found["ocr"])
manifest=parse_manifest(found["split"])
master=manifest.merge(labels,on="record_key",validate="one_to_one").merge(ocr,on="record_key",validate="one_to_one")

if len(master)!=9318:
    raise RuntimeError(f"aligned rows={len(master)} != 9318")
counts={k:int(v) for k,v in master.split.value_counts().to_dict().items()}
if counts!={"train":7454,"validation":932,"test":932}:
    raise RuntimeError(f"split-count drift {counts}")

payload="\n".join(
    f"{int(r.official_row_index)}\t{r.image_id}\t{r.split}"
    for r in master.sort_values("official_row_index").itertuples()
)+"\n"
row_sha=hashlib.sha256(payload.encode()).hexdigest()
if row_sha!=EXPECTED_CANONICAL_ROW_SHA256:
    raise RuntimeError(f"canonical-row SHA drift {row_sha}")

test_payload=[]
for r in master[master.split.eq("test")].sort_values("official_row_index").to_dict("records"):
    tar=[x for x in TARGET_LABELS if x in set(r["target_list"])]
    test_payload.append({
        "sample_id":str(int(r["official_row_index"])),
        "image":Path(str(r["image_id"])).name.lower(),
        "offensive":r["offensive"],"vulgar":r["vulgar"],"abuse":r["abuse"],"sarcastic":r["sarcastic"],
        "target":tar if tar else ["None"],
        "language":r["language"],
    })
test_sha=hashlib.sha256(
    json.dumps(test_payload,ensure_ascii=False,sort_keys=True,separators=(",",":")).encode()
).hexdigest()
if test_sha!=EXPECTED_TEST_IDENTITY_SHA256:
    raise RuntimeError(f"ordered TEST identity SHA drift {test_sha}")

# Resolve every canonical image and verify full image-bundle SHA before TEST.
valid_ext={".jpg",".jpeg",".png",".webp",".bmp"}
by_key={}
for p in INPUT_ROOT.rglob("*"):
    try:
        if p.is_file() and p.suffix.casefold() in valid_ext:
            by_key.setdefault(norm_key(p.name),[]).append(p)
    except OSError:
        pass

image_map={}
for key,name in zip(master.record_key,master.image_id):
    cand=by_key.get(key,[])
    exact=[p for p in cand if p.name.casefold()==Path(str(name)).name.casefold()]
    chosen=exact[0] if len(exact)==1 else (cand[0] if len(cand)==1 else None)
    if chosen is None:
        raise RuntimeError(f"missing/ambiguous image {name}: {[str(x) for x in cand[:5]]}")
    image_map[key]=str(chosen)
master["image_path"]=master.record_key.map(image_map)

h=hashlib.sha256()
req=master.sort_values("image_id")[["image_id","image_path"]].drop_duplicates().to_dict("records")
for i,r in enumerate(req,1):
    name=Path(str(r["image_id"])).name
    h.update(name.encode()+b"\0"+sha256_file(r["image_path"]).encode()+b"\n")
    if i%500==0 or i==len(req):
        print(f"[IMAGE SHA] {i}/{len(req)}",flush=True)
image_sha=h.hexdigest()
if image_sha!=EXPECTED_IMAGE_BUNDLE_SHA256:
    raise RuntimeError(f"image-bundle SHA drift {image_sha}")

master["target_json"]=master.target_list.map(lambda x:json.dumps(x,separators=(",",":")))
PREPARED_MANIFEST=WORK_ROOT/"prepared_manifest_FINAL_TEST.parquet"
master[[
    "official_row_index","image_id","image_path","split","cleaned_ocr",
    *BINARY_TASKS,"language","target_json"
]].to_parquet(PREPARED_MANIFEST,index=False)

print("[CANONICAL DATASET + TEST IDENTITY PASS]", {
    "rows":9318,
    "split_counts":counts,
    "ordered_test_sha256":test_sha,
    "image_bundle_sha256":image_sha,
    "TEST_model_forwards":0,
})


In [ ]:
# CELL 5 — FIVE-SEED DEVELOPMENT FREEZE + VALIDATION REPLAY
# Still NO TEST backbone forward.
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    f1_score, accuracy_score, average_precision_score,
    hamming_loss, precision_score, recall_score
)

class ResidualFacetAdapter(nn.Module):
    def __init__(self, hidden=1024, bottleneck=256, dropout=0.10):
        super().__init__()
        self.norm=nn.LayerNorm(hidden)
        self.block=nn.Sequential(
            nn.Linear(hidden,bottleneck), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(bottleneck,hidden), nn.Dropout(dropout),
        )
    def forward(self,x):
        return x + self.block(self.norm(x))

class FacetGateTarget(nn.Module):
    def __init__(self):
        super().__init__()
        D=1024
        self.product_norm=nn.LayerNorm(D)
        self.difference_norm=nn.LayerNorm(D)
        self.target_view_gate=nn.Sequential(nn.Linear(4*D,128),nn.GELU(),nn.Linear(128,4))
        self.target_expert_shared=nn.Sequential(nn.Linear(D,D),nn.GELU(),nn.Dropout(0.10))
        self.target_expert_adapter=ResidualFacetAdapter(D,256,0.10)
        self.target_expert_head=nn.Linear(D,7)
        self.target_selector=nn.Sequential(nn.Linear(6,16),nn.GELU(),nn.Linear(16,1))
    def forward(self,v,t,J,zG):
        if v.shape[-1]!=1024 or t.shape[-1]!=1024 or J.shape[-1]!=1024 or zG.shape[-1]!=7:
            raise RuntimeError(f"Facet interface shape drift: v={v.shape} t={t.shape} J={J.shape} zG={zG.shape}")
        prod=self.product_norm(v*t)
        diff=self.difference_norm(torch.abs(v-t))
        views=torch.stack([v,t,prod,diff],dim=1)
        gate_in=torch.cat([v,t,prod,diff],dim=-1)
        view_weights=F.softmax(self.target_view_gate(gate_in.float()),dim=-1)
        h=(views.float()*view_weights.unsqueeze(-1)).sum(dim=1)
        zH=self.target_expert_head(self.target_expert_adapter(self.target_expert_shared(h)))
        pG=torch.sigmoid(zG.float()); pH=torch.sigmoid(zH.float())
        d_it=(1.0-F.cosine_similarity(v.float(),t.float(),dim=-1)).clamp(0,2).unsqueeze(-1).expand(-1,7)
        sf=torch.stack([pG,pH,(pH-pG).abs(),(pG-.5).abs(),(pH-.5).abs(),d_it],dim=-1)
        selector_logit=self.target_selector(sf.reshape(-1,6)).reshape(-1,7)
        selector=torch.sigmoid(selector_logit)
        zStar=zG.float()+selector*(zH.float()-zG.float())
        return zStar,{
            "zH":zH,
            "selector_logit":selector_logit,
            "selector":selector,
            "view_weights":view_weights,
            "image_text_disagreement":d_it[:,0],
        }

def target_metric_bundle(y,prob,thr,full=False):
    y=np.asarray(y,dtype=np.int64)
    p=np.asarray(prob,dtype=np.float64)
    th=np.asarray(thr,dtype=np.float64).reshape(1,-1)
    pred=(p>=th).astype(np.int64)
    per_f1=[float(f1_score(y[:,j],pred[:,j],zero_division=0)) for j in range(7)]
    per_ap=[]
    per_detail={}
    for j,lab in enumerate(TARGET_LABELS):
        ap=float(average_precision_score(y[:,j],p[:,j])) if len(np.unique(y[:,j]))>=2 else float("nan")
        per_ap.append(ap)
        if full:
            per_detail[lab]={
                "support":int(y[:,j].sum()),
                "precision":float(precision_score(y[:,j],pred[:,j],zero_division=0)),
                "recall":float(recall_score(y[:,j],pred[:,j],zero_division=0)),
                "f1":per_f1[j],
                "ap":ap,
                "threshold":float(th[0,j]),
            }
    out={
        "macro_f1":float(np.mean(per_f1)),
        "map":float(np.nanmean(per_ap)),
        "micro_f1":float(f1_score(y,pred,average="micro",zero_division=0)),
        "samples_f1":float(f1_score(y,pred,average="samples",zero_division=0)),
        "exact_set_accuracy":float(np.mean(np.all(y==pred,axis=1))),
        "hamming_loss":float(hamming_loss(y,pred)),
        "per_label_f1":dict(zip(TARGET_LABELS,per_f1)),
        "per_label_ap":dict(zip(TARGET_LABELS,per_ap)),
        "rare_target_macro_f1":float(np.mean([
            per_f1[TARGET_LABELS.index("National")],
            per_f1[TARGET_LABELS.index("Religious")]
        ])),
    }
    if full:
        out["per_label"]=per_detail
    return out, pred

def normalized_parent_thresholds(path):
    obj=json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(obj,dict) and isinstance(obj.get("thresholds"),dict):
        obj=obj["thresholds"]
    if isinstance(obj,dict) and isinstance(obj.get("binary"),dict) and isinstance(obj.get("target"),dict):
        out={"binary":{t:float(obj["binary"][t]) for t in BINARY_TASKS},
             "target":{lab:float(obj["target"][lab]) for lab in TARGET_LABELS}}
        return out
    raise RuntimeError(f"Unexpected normalized Gated threshold schema: {path}")

master_val=master[master.split.eq("validation")].sort_values("official_row_index").reset_index(drop=True)
val_gold_bin=np.asarray([[1 if str(r[t])=="Yes" else 0 for t in BINARY_TASKS] for r in master_val.to_dict("records")],dtype=np.int64)
val_gold_lang=np.asarray([LANGUAGE_LABELS.index(str(x)) for x in master_val.language],dtype=np.int64)
val_gold_target=np.asarray([
    [1 if lab in set(json.loads(s)) else 0 for lab in TARGET_LABELS]
    for s in master_val.target_json
],dtype=np.int64)

dev_rows=[]
for seed in PUBLICATION_SEEDS:
    root=Path(FACET[seed]["root"])
    fc=np.load(root/"facetgate_feature_cache.npz",allow_pickle=False)
    split=fc["split"].astype(str)
    vm=split=="validation"
    if int(vm.sum())!=932:
        raise RuntimeError(f"Seed {seed} validation-cache rows != 932")
    idx=fc["official_row_index"][vm].astype(np.int64)
    if not np.array_equal(idx, master_val.official_row_index.to_numpy(dtype=np.int64)):
        raise RuntimeError(f"Seed {seed} validation row identity/order mismatch")

    V=fc["visual"][vm].astype(np.float32)
    T=fc["text"][vm].astype(np.float32)
    J=fc["joint"][vm].astype(np.float32)
    ZG=fc["z_target"][vm].astype(np.float32)
    ZB=fc["z_binary"][vm].astype(np.float32)
    ZL=fc["z_language"][vm].astype(np.float32)
    Y=fc["y_target"][vm].astype(np.int64)
    if not np.array_equal(Y,val_gold_target):
        raise RuntimeError(f"Seed {seed} validation Target gold mismatch")

    state=torch.load(root/"selected_checkpoint.pt",map_location="cpu",weights_only=False)
    facet=FacetGateTarget().cpu().eval()
    facet.load_state_dict(state["facetgate_state"],strict=True)

    pf=[]
    with torch.no_grad():
        for st in range(0,932,128):
            sl=slice(st,min(st+128,932))
            zs,_=facet(
                torch.from_numpy(V[sl]),torch.from_numpy(T[sl]),
                torch.from_numpy(J[sl]),torch.from_numpy(ZG[sl])
            )
            pf.append(torch.sigmoid(zs).numpy())
    PF=np.concatenate(pf,axis=0)

    fthr=np.asarray([FACET[seed]["facet_thresholds"]["thresholds"][lab] for lab in TARGET_LABELS],dtype=np.float64)
    fm,_=target_metric_bundle(Y,PF,fthr)

    # Compare the replay to the frozen published validation result.
    archived=FACET[seed]["facet_result"]["target_facetgate_calibrated"]
    for key in ["macro_f1","map","micro_f1","samples_f1","exact_set_accuracy","hamming_loss","rare_target_macro_f1"]:
        if abs(float(fm[key])-float(archived[key]))>2e-5:
            raise RuntimeError(
                f"Seed {seed} validation FacetGate replay mismatch {key}: "
                f"replay={fm[key]} archived={archived[key]}"
            )

    # Reproduce frozen Gated Core5 from the same validation interface.
    pthr=normalized_parent_thresholds(root/"parent_gated_thresholds.normalized.json")
    bin_prob=1.0/(1.0+np.exp(-np.clip(ZB.astype(np.float64),-60,60)))
    core_acc=[]; core_mf1=[]
    for k,t in enumerate(BINARY_TASKS):
        pred=(bin_prob[:,k]>=float(pthr["binary"][t])).astype(np.int64)
        core_acc.append(float(accuracy_score(val_gold_bin[:,k],pred)))
        core_mf1.append(float(f1_score(val_gold_bin[:,k],pred,average="macro",zero_division=0)))
    lp=np.exp(ZL.astype(np.float64)-ZL.max(axis=1,keepdims=True))
    lp/=lp.sum(axis=1,keepdims=True)
    lpred=lp.argmax(axis=1)
    core_acc.append(float(accuracy_score(val_gold_lang,lpred)))
    core_mf1.append(float(f1_score(val_gold_lang,lpred,average="macro",zero_division=0)))
    c5a=float(np.mean(core_acc)); c5f=float(np.mean(core_mf1))
    archived_core=FACET[seed]["facet_result"]["core5"]
    if abs(c5a-float(archived_core["accuracy"]))>2e-5 or abs(c5f-float(archived_core["macro_f1"]))>2e-5:
        raise RuntimeError(
            f"Seed {seed} frozen Gated Core5 validation replay mismatch "
            f"{c5a}/{c5f} vs {archived_core}"
        )

    delta_pp=float(FACET[seed]["facet_result"]["delta"]["target_macro_f1_pp"])
    core_delta=float(FACET[seed]["facet_result"]["core5"].get("delta_vs_gated",0.0))
    dev_rows.append({
        "seed":seed,
        "gated_step":int(FACET[seed]["base_binding"]["selected_gated_step"]),
        "facetgate_step":int(state["step"]),
        "validation_target_delta_pp":delta_pp,
        "validation_core5_delta":core_delta,
        "validation_target_macro_f1":float(fm["macro_f1"]),
        "validation_core5_macro_f1":c5f,
    })
    print("[VALIDATION REPLAY PASS]",dev_rows[-1])

dev_df=pd.DataFrame(dev_rows).sort_values("seed").reset_index(drop=True)
mean_delta=float(dev_df.validation_target_delta_pp.mean())
positive_count=int((dev_df.validation_target_delta_pp>0).sum())
core_identity=bool((dev_df.validation_core5_delta.abs()<=1e-10).all())

FIVE_SEED_GATE={
    "mean_calibrated_target_macro_f1_improvement_ge_1pp":mean_delta>=1.0,
    "positive_target_delta_at_least_4_of_5":positive_count>=4,
    "core5_predictions_remain_matched_gated":core_identity,
    "common_protocol_unchanged":True,
}
if not all(FIVE_SEED_GATE.values()):
    atomic_json(WORK_ROOT/"FIVE_SEED_DEVELOPMENT_GATE_FAIL.json",{
        "rows":dev_rows,"mean_delta_pp":mean_delta,"positive_count":positive_count,"gate":FIVE_SEED_GATE
    })
    raise RuntimeError(
        "Frozen FacetGate five-seed promotion criterion FAILED. "
        "Final TEST remains unopened. " + json.dumps(FIVE_SEED_GATE)
    )

dev_df.to_csv(WORK_ROOT/"qwen3_facetgate_5seed_validation_freeze.csv",index=False)
atomic_json(WORK_ROOT/"FIVE_SEED_DEVELOPMENT_GATE_PASS.json",{
    "rows":dev_rows,
    "mean_target_macro_f1_delta_pp":mean_delta,
    "positive_seed_count":positive_count,
    "gate":FIVE_SEED_GATE,
    "TEST_model_forwards":0,
})
print("[FIVE-SEED DEVELOPMENT GATE PASS]",{
    "mean_delta_pp":mean_delta,
    "positive_seed_count":positive_count,
    "Core5_identity":core_identity,
    "TEST_forwards":0,
})


In [ ]:
# CELL 6 — PIN EXACT QWEN MODEL REVISION ONCE FOR ALL FIVE TEST SEEDS
# TEST still has not been forwarded.
import os
from pathlib import Path
from huggingface_hub import snapshot_download

token=None
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token=os.environ.get("HF_TOKEN")

MODEL_DIR=Path("/kaggle/temp")/f"qwen3_facetgate_final_{MODEL_REVISION[:12]}"
MODEL_DIR.mkdir(parents=True,exist_ok=True)

if not any(MODEL_DIR.iterdir()):
    print("[MODEL SNAPSHOT] downloading pinned revision",MODEL_ID,MODEL_REVISION,flush=True)
    snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
        token=token,
        local_dir_use_symlinks=False,
    )

# model index provenance
model_index=MODEL_DIR/"model.safetensors.index.json"
if not model_index.is_file():
    raise RuntimeError("Pinned Qwen model snapshot lacks model.safetensors.index.json")

print("[PINNED QWEN SNAPSHOT PASS]",{
    "path":str(MODEL_DIR),
    "model_id":MODEL_ID,
    "revision":MODEL_REVISION,
    "TEST_forwards":0,
})


In [ ]:
# CELL 7 — EMBEDDED EXACT QWEN LOCKED-TEST FEATURE WORKER
# Derived execution-only from the successful FacetGate Qwen development feature worker.
# Scientific feature extraction is unchanged; only the allowed split is TEST.

QWEN_DEV_FEATURE_WORKER_SHA256 = 'e3e6abe582eec0cd356493ac229551ca668affbaaa6f456d8563b1e50d0cfd91'
QWEN_LOCKED_TEST_FEATURE_WORKER_SHA256 = '5b2d90aaf41c65d850ca6e252a4672d8b5ca704a9657074b7544837da9f2f517'
QWEN_LOCKED_TEST_FEATURE_WORKER_ZLIB_B64 = 'eNqtWuty2zqS/q+nQLC1U2QCM76dyyiDrXJsOZNzHDvrKJnaqFQsioQkxryFF1m2x1v7EPOE8yTb3QBvkuyz58ymkogEGo1Go/vrboCDeZ7GzHXnVVnlynVZGGdpXjIvSdLSK8M0KQamKS0E+1akiWC5lwRpLFjmlcsonAm29Ar9sPBr6qSKszvmFSzJkDAJ4BH+ZoFgZZr7y5qOXpwkIdJks9WZV4mPUngREpwPSNyP7y9qOd/H3kIJ/XMeRmrQPDkXVydn7vj68+XpyXh05r7/cPJu9EmO80oNBqfn7ySuxYlSLygssxDnI/xaaeGoZBXmaTLh5yeno7H7t6vrX0fX7unV5fn7d3xqO7nyArdU69Ky7UE4Z6ArBiydhSotPlce6XIWJkGYLNxi6R3+8CO3hwMGf3IvLBS7rpIyjNUoz9Pc4h/CogBK9p+3KmFq7fnlXublKimZ4cUML/bpryfcHlyfXP4qw6TsiYqNINsbdnF1enLh7qRpu4gS1nVxtkVEre6n919HQISro4YX8nC4S/g5P/d8Vb7zStVIq9a0N7n6XoW5Ktj4eH3IXrOWsTwEUwGdPVDTIyzpbPTl/elI6n0P1Cr0FbD2q8AbPrRSA+UbYxvY5RSqdA2xZmAPtG1Cjwos2JIJxyc+fYXDYXCSOb9BodnHXlJ5kbubqLPlE+5FUXrrRql/o9AmipJPdyqKn4doxZqQjUefxuw2zW9U3uoJ9BaFfgjeV5XLNA/vyQFBO/Pci5XMAm14YBvfK7A0mj7LFdpK4ILE4Zymt50C1O+uvKhShcXT+RyYwmJykBPsSK05WjDqjt6sIE8z8gt7gPJLmm1C/zsFSFQ66rvFaWn29I/zBqVFKrGIr/1C/vno4GeW5gzNz+pMJiUvQX2wbKeoYnCwF/Kn4x+OnyAFQcLAqKmh//PRIZLjbCi1bnna/U69JE1CHzaHZHrNcL49GIL/GM3EgjyclyAAmz+xi5wWOEvTyKJJtYQvpNGb4yV3INvTQlymyR6xA00ycH0Fe7pjsr6LwaTY6n64OhtduOdX1387uT4DnPp8OZb7Aw2WoMykmKd5rPKiRs2Tqkw/5qmvCthNwd6GZXGSBG/vQNbTNJmHC0FYdPTl4kMaqEgzyhRqQDP4CM+6K6vZyB5TB4e4YJy0l7UTxTjERbTlUwHLAsOZA1QXbppEd2QmosyronRzFaelcn2gl+deVID5zLxCya5UW1OQbv/FeQQx+V55sDfa/aATNSK3lWRh8AA7d49nYIvEdZbM6M0lDm55lynJk/kx13zxT0NSFcoN0moWKU29wcFP46wC2QJiolFpDjOWBz/amptGPnD8TD5wPizKvEbBR0H0O8cOSDmy2cEtNaKihYa2wMvAEGtFhoVLNB6IbHblDSNujgI/tOzBMgwClVBE0e0QEF3k59KrZTtamQ7GTqNYRw9yi/Beg4R+f0FcSAywdeWXAHG6BwDuiUhE8VMTGYd90G8YYRpDdcr0RiUwW+5kXqBDdAj7z/NwsSz5IMxkSxtiOuE274PiXgahX1qwMK8EhYeZ4Cg5Fw+PNoIO/LxhXgbOrwKtJFwTLwDPEQlcFSwURyAp7iHJChiP0mSx0aFRIswc5CwfXr4s7sUGhyGkFC8Pf4a/os9h+MPBoe54bOWgpEeBLJozoAHzxGqFs1kWj8GEs3AN/Vw0bG0BHd666WjY1iBGIs4x9asV4cG+FK1ecAZ7QwYTOE3b7jhJuxjcJV4c+ns0PQNLAcvzS3ioooBYzFTNBZGXdon21Q0DuWurgcVKQeSiFjDkFAghgP3l72aHveDv/6ExvM+MhQUDbFZ1/On32n/ZfyauNMJCREyjldK5nYbWvS8XeiZGvNj7M5g9UHPAfkQjC8KAUfS/sRENO8/TeyA8TeM4TRgmXAFbHTgHbA5IEoDmcwjEBrQcM/L9h49X1+OTy/GQBSmJEiZAWLKr02uIbF7CYg+jV8GWEHAg+VkqZlJPPQH5LUsU6I4V3i0SxJr5bVgudd7tpBlEWhB4wmtlIl7YmLGHccdasFOGcb0XFr9+95YjJmQQG4ms1muxcw+NahDoaDY/Uh4glpv6OUwnwJndAqAC8xG9zXXsQNZAtJNpoIhpM/NkePTz8VQUN2G2yU3nMcisqCDxWUt+qiUgdSKmDRl/ZcFM6Bnw44CoYWbZDH2P8QmSjT58HP/XP//nH58/jRjVJFNu1g7jJQFe6RU3EFdgcEXFD+SdekKii4uFnDzwPI3A2znsfc4FRw+BXeND6EHQhx5SN3Tp3yH9PIqmG6eDXvoZ4v+P08cpTbDu6Al97M71lx54joqzCIzCajYUJBG1InX0wi1YqETlOnQCnzgrTWSFCFPliYvwqYlNA8gNMxWSZwC+yFarY+1kaWbxyJsR/qAPApqYVr1buBLcMtNLwwB9QA8wuVxDAKtfIEgWN3zqUH6mCUNskg0F+xODAWECMZdYTjHp3eHwds8ne75CwJXBDFipIfchQ7+50zIFCrEg2es4fe2l4yWATJj4URVAFYA+2Nn812A0r2vdk5FoGImougYER/q5xgYtjmZbbq3vv2nJAwPcBAbYoFNTxDdsKtum1nOfhGnV1QKt7DVJiEwYyFzeGePWe83WgqYUNAvBnYn+EJJuvTyw1mbWOF1BzHi4Ga7AVU1aI0B97gyzYVCwdkZciE5vMDkhQ7JWxt1WFOduBIW5tROCDgvLfmzByxwypO4i9wLL1jmTA7UXJIwFoBOWmFzsyqFazaRVKXVu8/Ilybxt5pjn+Z6/VF00MvqA4U4Ek7l1EgTRUwli9LT1DgY+jCnYiU7PrCRxIJOrImXkQrW6gB+QRLpWoaK5WIqZhMguArnvHOx3xC+qDBjYTkMOLoYjQCt5LIHxhXen8kt4sZZ2OwopaCOQ5BOUrygnZIA4AHJIL7dgRlvA67vRxWeLns6gEoTlWgG9GbKZWG502s0SapOgFawx5zM29Kqd32qEBcsBzeTerdkqzM11+riEmrmoc1hIlrFWJ3iS3M8qLm4V5n2mNjBbhGlAgU7oJT7EYu9W4H7alLH1O/SZTxAWfh5CIgWMV4rbmhzzm1sJ/yabBFPIxn7vHGQbZFc7+Hc6p4NP45PxCNsHufouJ3wF7DQaf1M6oGjc77eAGUP+UN65C2DFMeHEcwUkxXBkaoGirmT4DNaS37mkXSCKvGRRIVJiAw3KMfk3b7E+4XK1JJ0GCj/TgXmVkzU57RpdFmRH916btIXRqqa4NYb6uSpA65mRo4TzELMX1FA9VD6Yh0c88IH6KQrB42kGGxKA/SmdScmD/cPjwZb2dhh96yXkx10T1y3irOMOdotpeOTS24j/X+a9PX3SWQ9fnhkWn8JFnIbBBhdtCc9JdtYV6qwnT9e9CX16rGuzkg2InWF59bAe1uh2hjWJoJGtbZBjGwtE84R4/WrS2GBjfnz62Juta7PbM3ZXcPD8ZH22PduXXTZH/a1ufaJH9VOPCueF4gfDQIDTT8jhrF1evNWEVduWZ280mNMDa9Phe6/IqAYA/Uus+0hQPzUc+5jQfcPRGxDRe9XcO5DRebEH06HJumeSfHSC+tHJqvFjUSX1MQHGY4J/twVFC4YKTMb9shuFWyzB9Kdl8RyyPODcjzgu9kp/uY0oHT7yoX1+NJkQSmeOS+paPMNtxnZIpIA5qhSyL5Y59ckwZSiuZSQfQPFTSDpkDCGWTfBMejiko/RpXUxdykjXZAVEdANkX2SSOVDypIVlXZKPZjqjOToEovGzvb+cXF8/QzD4+rbXe7wx/OtFr/tos7s/+U993rCKMFi3FJfYC6n5j8cwNIwXWC5Opm/0SS0+kgt9E5YrsIpG5aqkirEoobq6wFwwx4fmRBYMoakqiQ3Uk/XZ7fC3z2tzhfndrvPameffzPD4wGQzxgh6ebDslPzUuxSQ+smtzJj6njrvZa8kOzCFjbR0Yg+gUl+m2H8ClrZTAHyre2Xt40UH0JW/SdcvFzZrBVMo7NQQpf+svR2q8YWKA6OH5UouJ/uwdytXR2y5XE3CeKo3HxLTGOprkKKDjyB4h7Z8lvb35vqKzjbNuV0nTV7JLaS1aonr6VEwuQG1VrlJ1LCkgNyDXMtcbAHdZCXKKeR4sdw7sLfH0oGPxDEvV6+sA2d/jyD7ZdlQ/CI1aINuLaJumGC9qau2FQxkvj5NCvHCYQ5bNWRNUCemAfuWQidbheq2zf+Xhv0W75a7qUgJMnV5nMGAkNRCF8dtcamyQhKqDuuAovHdKpY6CuMbenHd/diMvZ+ZdB+gHqqBSTfkTNZTC3nDb2vQe78R2Butt1NEshepNM8215h2SMEA2rBlCOtMRJN9mXybyhU4G8i7xPPwrIL/6bIcfsEUodi0uuBHPo+Dyt85CBEbx/3yO8d9fYuj7mf/92F44bf0oO3YcLggDtEf4HBkONCK78s/wOEnU19TzEAueIZDuL7jrnLaBBA86FJJ0J4t6mOfMAAauw4tW0R1qGhQ0vr26sD+9x/2pdxHkIQ3KS8hkucoxZxTqGYPdJU91dDYul59bf4Aox5fP1w+cjGPKvC19tgRz2LXgsKDWK5EDUGihhkBuCF+Eei4kLUJNEBxPxP3kbgv+yIe7KOMw+5lOuG1PqmABAIPNnqfRZCz1N82BCGeuL6ecxS8dE1z4eZecqOX5yTZPR/ADhXeSt3TbRYQAFpYwFls74XUOyZqvWO49wovz707S2+RLfSlb6dDb4vdXq51/wBkV14kv1D+K8eCgEyiW4h7V3u9/PoWnmtXll8v4E37q/w6huoJrE8+8AI0Ent8yD8cXc3nYABuE9Tc1YG7715cnf46OnMpNNcaQoDEyhc1woeoEsFv0zwKuM7UoAcSED68bIQ3t5ZgcsPOJSZaoDDPuTL1c5egaQQy+khh2PleoWWuz+hcWIp/k6EmXEhWFion0zRjnqfpMHviE5dhz0Y2OjvD9Slr78ICtKvvN/T1BkWMPbrdMOGE7jj0efIbSD62LjDaG1ZdAGGS4WZpFPp3wLt/lGbOQhdKv1lq91mu3eEJroSfNST6myiw/5gPMakWlCPq6806VYNdfSpLw8u65kMRt/7MAzcNnfxxYP1LLodfNHHbuc0hv9WfJ9E3TgGgZmGhMQt0NXCCQ1tQzol1C6/K+d7PdEGKIMUn9M0TOx+djD9fj/CLo+szdnr14ePFaDyacrHB0u6B1P8CMJruMQ=='

import base64, zlib, hashlib
raw=zlib.decompress(base64.b64decode(QWEN_LOCKED_TEST_FEATURE_WORKER_ZLIB_B64))
observed=hashlib.sha256(raw).hexdigest()
if observed!=QWEN_LOCKED_TEST_FEATURE_WORKER_SHA256:
    raise RuntimeError("Embedded final TEST worker SHA mismatch")
WORKER_SOURCE=raw.decode("utf-8")
compile(WORKER_SOURCE,"qwen_facetgate_locked_test_feature_worker.py","exec")
print({
    "development_worker_sha256":QWEN_DEV_FEATURE_WORKER_SHA256,
    "locked_test_worker_sha256":QWEN_LOCKED_TEST_FEATURE_WORKER_SHA256,
    "compile":"PASS",
    "TEST_forwards_before_execution":0,
})


In [ ]:
# CELL 8 — FINAL LOCKED TEST: ALL FIVE FROZEN SEEDS
# This is the ONLY cell that performs Qwen TEST model forwards.
import os, sys, subprocess, shutil, time, json, hashlib
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, accuracy_score

if not AUTHORIZE_LOCKED_TEST or LOCKED_TEST_ACK!="I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST":
    raise RuntimeError("Final TEST authorization missing.")
if not all(FIVE_SEED_GATE.values()):
    raise RuntimeError("Five-seed development gate is not frozen PASS.")

import torch as _torch
if not _torch.cuda.is_available() or _torch.cuda.device_count()!=2:
    raise RuntimeError(
        f"Final Qwen FacetGate TEST requires Kaggle T4x2. "
        f"cuda={_torch.cuda.is_available()} count={_torch.cuda.device_count()}"
    )

WORKER_PATH=WORK_ROOT/"qwen_facetgate_locked_test_feature_worker.py"
WORKER_PATH.write_text(WORKER_SOURCE,encoding="utf-8")
if sha256_file(WORKER_PATH)!=QWEN_LOCKED_TEST_FEATURE_WORKER_SHA256:
    raise RuntimeError("Materialized final TEST worker SHA mismatch")

test_master=master[master.split.eq("test")].sort_values("official_row_index").reset_index(drop=True)
expected_test_idx=test_master.official_row_index.to_numpy(dtype=np.int64)

test_gold_bin=np.asarray(
    [[1 if str(r[t])=="Yes" else 0 for t in BINARY_TASKS] for r in test_master.to_dict("records")],
    dtype=np.int64
)
test_gold_lang=np.asarray([LANGUAGE_LABELS.index(str(x)) for x in test_master.language],dtype=np.int64)
test_gold_target=np.asarray([
    [1 if lab in set(json.loads(s)) else 0 for lab in TARGET_LABELS]
    for s in test_master.target_json
],dtype=np.int64)

def load_test_shards(seed, feature_dir):
    parts=[]
    total_forwards=0
    metas=[]
    for rank in [0,1]:
        npz=feature_dir/f"facet_features_rank{rank}.npz"
        js=feature_dir/f"facet_features_rank{rank}.json"
        if not npz.is_file() or not js.is_file():
            raise RuntimeError(f"Missing TEST shard seed={seed} rank={rank}")
        meta=json.loads(js.read_text(encoding="utf-8"))
        if meta.get("schema")!="M3Offend_FacetGate_v1_0_LOCKED_TEST_feature_shard":
            raise RuntimeError(f"Wrong TEST shard schema seed={seed} rank={rank}")
        if int(meta.get("rank",-1))!=rank or int(meta.get("world",-1))!=2:
            raise RuntimeError(f"TEST shard rank/world drift seed={seed} rank={rank}")
        if int(meta.get("seed",-1))!=seed:
            raise RuntimeError(f"TEST shard seed drift expected={seed} got={meta.get('seed')}")
        if meta.get("model_id")!=MODEL_ID or meta.get("model_revision")!=MODEL_REVISION:
            raise RuntimeError(f"TEST shard model identity drift seed={seed}")
        if meta.get("parent_checkpoint_fingerprint")!=FACET[seed]["base_binding"]["checkpoint_fingerprint"]:
            raise RuntimeError(f"TEST shard Gated parent fingerprint drift seed={seed}")
        if meta.get("feature_binding_sha256")!=QWEN_PARENT_INTERFACE_SHA256:
            raise RuntimeError(f"TEST shard Qwen interface binding drift seed={seed}")
        if int(meta.get("representation_dim",-1))!=1024 or not bool(meta.get("locked_test_authorized",False)):
            raise RuntimeError(f"TEST shard contract drift seed={seed}")
        total_forwards += int(meta.get("test_model_forwards",-1))
        metas.append(meta)
        with np.load(npz,allow_pickle=False) as z:
            parts.append({k:z[k] for k in z.files})

    merged={}
    for key in ["official_row_index","image_id","split","visual","text","joint","z_binary","z_language","z_target"]:
        merged[key]=np.concatenate([p[key] for p in parts],axis=0)
    order=np.argsort(merged["official_row_index"].astype(np.int64),kind="stable")
    for key in merged:
        merged[key]=merged[key][order]

    if total_forwards!=932:
        raise RuntimeError(f"Seed {seed} TEST model-forward count={total_forwards}, expected exactly 932")
    if len(merged["official_row_index"])!=932 or len(np.unique(merged["official_row_index"]))!=932:
        raise RuntimeError(f"Seed {seed} TEST shard row duplication/loss")
    if not np.array_equal(merged["official_row_index"].astype(np.int64),expected_test_idx):
        raise RuntimeError(f"Seed {seed} ordered TEST identity mismatch after worker")
    if not np.all(merged["split"].astype(str)=="test"):
        raise RuntimeError(f"Seed {seed} non-TEST row in final feature cache")
    return merged,total_forwards,metas

def core5_metrics(zb,zl,parent_thr):
    bp=1.0/(1.0+np.exp(-np.clip(zb.astype(np.float64),-60,60)))
    task_acc=[]; task_mf1=[]; tasks={}; preds={}
    for k,t in enumerate(BINARY_TASKS):
        th=float(parent_thr["binary"][t])
        pr=(bp[:,k]>=th).astype(np.int64)
        a=float(accuracy_score(test_gold_bin[:,k],pr))
        f=float(f1_score(test_gold_bin[:,k],pr,average="macro",zero_division=0))
        task_acc.append(a); task_mf1.append(f)
        tasks[t]={"accuracy":a,"macro_f1":f,"threshold":th}
        preds[t]=pr
    lp=np.exp(zl.astype(np.float64)-zl.max(axis=1,keepdims=True))
    lp/=lp.sum(axis=1,keepdims=True)
    lpred=lp.argmax(axis=1)
    la=float(accuracy_score(test_gold_lang,lpred))
    lf=float(f1_score(test_gold_lang,lpred,average="macro",zero_division=0))
    task_acc.append(la); task_mf1.append(lf)
    tasks["language"]={"accuracy":la,"macro_f1":lf}
    return {
        "accuracy":float(np.mean(task_acc)),
        "macro_f1":float(np.mean(task_mf1)),
        "tasks":tasks,
    },bp,lp,preds,lpred

PER_SEED_RESULTS=[]
TOTAL_TEST_FORWARDS=0

for seed in PUBLICATION_SEEDS:
    print("\n"+"="*100)
    print(f"FINAL LOCKED TEST — QWEN3 FACETGATE — SEED {seed}")
    print("="*100)

    seed_dir=WORK_ROOT/f"seed_{seed}"
    feature_dir=seed_dir/"locked_test_features"
    seed_dir.mkdir(parents=True,exist_ok=True)
    feature_dir.mkdir(parents=True,exist_ok=True)

    facet_root=Path(FACET[seed]["root"])
    gated_cp=Path(GATED[seed]["path"])
    parent_thr=normalized_parent_thresholds(facet_root/"parent_gated_thresholds.normalized.json")

    lock_payload={
        "dataset_test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
        "seed":seed,
        "model_id":MODEL_ID,
        "model_revision":MODEL_REVISION,
        "gated_step":int(FACET[seed]["base_binding"]["selected_gated_step"]),
        "gated_inference_fingerprint":FACET[seed]["base_binding"]["checkpoint_fingerprint"],
        "facetgate_step":int(FACET[seed]["selected_facet_step"]),
        "facetgate_checkpoint_sha256":FACET[seed]["selected_facet_sha256"],
        "facetgate_validation_threshold_sha256":FACET[seed]["facet_threshold_sha256"],
        "facetgate_protocol_sha256":FACETGATE_COMMON_SHA256,
        "gated_protocol_sha256":PARENT_GATED_PROTOCOL_SHA256,
        "qwen_feature_binding_sha256":QWEN_PARENT_INTERFACE_SHA256,
    }
    lock_id=json_sha(lock_payload)
    marker=seed_dir/"LOCKED_TEST_ONCE_MARKER.json"

    reuse=False
    if marker.is_file():
        old=json.loads(marker.read_text(encoding="utf-8"))
        metrics_path=seed_dir/"locked_test_metrics.json"
        ledger_path=seed_dir/"locked_test_prediction_ledger.parquet"
        if (
            old.get("lock_id")==lock_id
            and int(old.get("test_rows",-1))==932
            and metrics_path.is_file() and ledger_path.is_file()
            and sha256_file(ledger_path)==old.get("ledger_sha256")
        ):
            reuse=True
            print("[REUSE IDENTICAL COMPLETED LOCK] no TEST forward repeated",lock_id)

    if not reuse:
        # Final per-seed worker configuration.
        cfg={
            "seed":seed,
            "model_id":MODEL_ID,
            "model_revision":MODEL_REVISION,
            "model_path":str(MODEL_DIR),
            "adapter_path":str(gated_cp/"adapter"),
            "heads_path":str(gated_cp/"classifier_heads.pt"),
            "prepared_manifest":str(PREPARED_MANIFEST),
            "feature_dir":str(feature_dir),
            "allow_locked_test":True,
            "task_instruction":TASK_INSTRUCTION,
            "expected_hidden":4096,
            "binary_tasks":BINARY_TASKS,
            "parent_checkpoint_fingerprint":FACET[seed]["base_binding"]["checkpoint_fingerprint"],
            "feature_binding_sha256":QWEN_PARENT_INTERFACE_SHA256,
        }
        cfgp=seed_dir/"locked_test_worker_config.json"
        cfgp.write_text(json.dumps(cfg,indent=2),encoding="utf-8")

        env=os.environ.copy()
        env["FACET_WORKER_CONFIG"]=str(cfgp)
        env["TOKENIZERS_PARALLELISM"]="false"

        cmd=[
            sys.executable,"-u","-m","torch.distributed.run",
            "--standalone","--nproc_per_node=2",str(WORKER_PATH)
        ]
        print("[TEST WORKER]", " ".join(cmd), flush=True)
        started=time.time()
        rc=subprocess.run(cmd,env=env,cwd=str(seed_dir)).returncode
        if rc!=0:
            raise RuntimeError(f"Seed {seed} final TEST feature worker failed rc={rc}")
        print(f"[TEST WORKER COMPLETE] seed={seed} seconds={time.time()-started:.1f}")

        M,seed_forwards,metas=load_test_shards(seed,feature_dir)
        TOTAL_TEST_FORWARDS += seed_forwards

        V=M["visual"].astype(np.float32)
        T=M["text"].astype(np.float32)
        J=M["joint"].astype(np.float32)
        ZB=M["z_binary"].astype(np.float32)
        ZL=M["z_language"].astype(np.float32)
        ZG=M["z_target"].astype(np.float32)

        # Core5 is the untouched frozen Gated output.
        c5,bp,lp,bpred,lpred=core5_metrics(ZB,ZL,parent_thr)

        # Same frozen Gated Target is retained as the causal comparator.
        pg=1.0/(1.0+np.exp(-np.clip(ZG.astype(np.float64),-60,60)))
        gthr=np.asarray([parent_thr["target"][lab] for lab in TARGET_LABELS],dtype=np.float64)
        gated_target,gated_target_pred=target_metric_bundle(test_gold_target,pg,gthr,full=True)

        # FacetGate Target only — no training/calibration/test selection.
        state=torch.load(facet_root/"selected_checkpoint.pt",map_location="cpu",weights_only=False)
        if int(state.get("step",-1))!=int(FACET[seed]["selected_facet_step"]):
            raise RuntimeError(f"Seed {seed} selected FacetGate state step drift")
        facet=FacetGateTarget().cpu().eval()
        facet.load_state_dict(state["facetgate_state"],strict=True)

        probs=[]; selectors=[]; views=[]
        with torch.no_grad():
            for st in range(0,932,128):
                sl=slice(st,min(st+128,932))
                zs,ex=facet(
                    torch.from_numpy(V[sl]),torch.from_numpy(T[sl]),
                    torch.from_numpy(J[sl]),torch.from_numpy(ZG[sl])
                )
                probs.append(torch.sigmoid(zs).numpy())
                selectors.append(ex["selector"].numpy())
                views.append(ex["view_weights"].numpy())
        pf=np.concatenate(probs,axis=0)
        sel=np.concatenate(selectors,axis=0)
        vw=np.concatenate(views,axis=0)

        fthr=np.asarray(
            [FACET[seed]["facet_thresholds"]["thresholds"][lab] for lab in TARGET_LABELS],
            dtype=np.float64
        )
        facet_raw,facet_raw_pred=target_metric_bundle(
            test_gold_target,pf,np.full(7,0.5,dtype=np.float64),full=True
        )
        facet_cal,facet_pred=target_metric_bundle(test_gold_target,pf,fthr,full=True)

        full6_acc=float((5.0*c5["accuracy"]+facet_cal["exact_set_accuracy"])/6.0)
        full6_mf1=float((5.0*c5["macro_f1"]+facet_cal["macro_f1"])/6.0)
        gated_full6_acc=float((5.0*c5["accuracy"]+gated_target["exact_set_accuracy"])/6.0)
        gated_full6_mf1=float((5.0*c5["macro_f1"]+gated_target["macro_f1"])/6.0)

        result={
            "schema":"M3Offend_Qwen3VL8B_FacetGate_v1_0_FINAL_LOCKED_TEST_SEED",
            "lock_id":lock_id,
            "seed":seed,
            "model_id":MODEL_ID,
            "model_revision":MODEL_REVISION,
            "test_rows":932,
            "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
            "test_model_forwards":932,
            "gated":{
                "selected_step":int(FACET[seed]["base_binding"]["selected_gated_step"]),
                "inference_fingerprint":FACET[seed]["base_binding"]["checkpoint_fingerprint"],
                "core5":c5,
                "target_calibrated":gated_target,
                "full6_accuracy":gated_full6_acc,
                "full6_macro_f1":gated_full6_mf1,
            },
            "facetgate":{
                "selected_step":int(FACET[seed]["selected_facet_step"]),
                "checkpoint_sha256":FACET[seed]["selected_facet_sha256"],
                "validation_threshold_sha256":FACET[seed]["facet_threshold_sha256"],
                "target_raw_0p5":facet_raw,
                "target_calibrated":facet_cal,
                "full6_accuracy":full6_acc,
                "full6_macro_f1":full6_mf1,
                "target_macro_f1_delta_vs_gated":float(facet_cal["macro_f1"]-gated_target["macro_f1"]),
                "target_macro_f1_delta_pp":float(100*(facet_cal["macro_f1"]-gated_target["macro_f1"])),
                "target_map_delta_vs_gated":float(facet_cal["map"]-gated_target["map"]),
                "selector_mean":float(sel.mean()),
                "selector_sd":float(sel.std()),
                "view_weight_mean":dict(zip(["visual","text","product","difference"],vw.mean(axis=0).tolist())),
            },
        }

        # Complete per-item ledger.
        ledger=pd.DataFrame({
            "official_row_index":expected_test_idx,
            "image_id":test_master.image_id.astype(str).to_numpy(),
            "seed":seed,
        })
        for k,t in enumerate(BINARY_TASKS):
            ledger[f"gold_{t}"]=test_gold_bin[:,k]
            ledger[f"gated_prob_{t}"]=bp[:,k]
            ledger[f"gated_pred_{t}"]=bpred[t]
            ledger[f"gated_threshold_{t}"]=float(parent_thr["binary"][t])
        ledger["gold_language"]=[LANGUAGE_LABELS[i] for i in test_gold_lang]
        ledger["gated_pred_language"]=[LANGUAGE_LABELS[i] for i in lpred]
        for k,lab in enumerate(LANGUAGE_LABELS):
            ledger[f"gated_prob_language_{lab}"]=lp[:,k]

        for j,lab in enumerate(TARGET_LABELS):
            ledger[f"gold_target_{lab}"]=test_gold_target[:,j]
            ledger[f"gated_target_prob_{lab}"]=pg[:,j]
            ledger[f"gated_target_pred_{lab}"]=gated_target_pred[:,j]
            ledger[f"gated_target_threshold_{lab}"]=float(gthr[j])
            ledger[f"facetgate_target_prob_{lab}"]=pf[:,j]
            ledger[f"facetgate_target_pred_{lab}"]=facet_pred[:,j]
            ledger[f"facetgate_target_threshold_{lab}"]=float(fthr[j])
            ledger[f"facetgate_selector_{lab}"]=sel[:,j]

        ledger["gated_target_prediction_json"]=[
            json.dumps([lab for j,lab in enumerate(TARGET_LABELS) if row[j]],ensure_ascii=False)
            for row in gated_target_pred
        ]
        ledger["facetgate_target_prediction_json"]=[
            json.dumps([lab for j,lab in enumerate(TARGET_LABELS) if row[j]],ensure_ascii=False)
            for row in facet_pred
        ]

        ledger_path=seed_dir/"locked_test_prediction_ledger.parquet"
        ledger.to_parquet(ledger_path,index=False)
        ledger.to_csv(seed_dir/"locked_test_prediction_ledger.csv",index=False)
        atomic_json(seed_dir/"locked_test_metrics.json",result)
        atomic_json(marker,{
            "schema":"M3Offend_Qwen3VL8B_FacetGate_TEST_ONCE_MARKER_v1_0",
            "lock_id":lock_id,
            "seed":seed,
            "test_rows":932,
            "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
            "test_model_forwards":932,
            "ledger_sha256":sha256_file(ledger_path),
            "metrics_sha256":sha256_file(seed_dir/"locked_test_metrics.json"),
        })

        # Free small inference objects before next seed.
        del facet,state,V,T,J,ZB,ZL,ZG,pf,pg,sel,vw,M
        import gc
        gc.collect()

    result=json.loads((seed_dir/"locked_test_metrics.json").read_text(encoding="utf-8"))
    PER_SEED_RESULTS.append(result)
    print("[SEED FINAL LOCKED TEST COMPLETE]",{
        "seed":seed,
        "Core5_MF1":result["gated"]["core5"]["macro_f1"],
        "Gated_Target_MF1":result["gated"]["target_calibrated"]["macro_f1"],
        "FacetGate_Target_MF1":result["facetgate"]["target_calibrated"]["macro_f1"],
        "FacetGate_Full6_MF1":result["facetgate"]["full6_macro_f1"],
        "test_rows":932,
    })

# Five seeds x 932 unique TEST items. If no prior identical marker was reused:
fresh_markers=[
    json.loads((WORK_ROOT/f"seed_{s}"/"LOCKED_TEST_ONCE_MARKER.json").read_text(encoding="utf-8"))
    for s in PUBLICATION_SEEDS
]
if any(int(m.get("test_rows",-1))!=932 for m in fresh_markers):
    raise RuntimeError("Per-seed final TEST marker row drift")
print("[ALL FIVE FINAL TEST SEEDS COMPLETE]")


In [ ]:
# CELL 9 — FIVE-SEED FINAL TEST AGGREGATION + PACKAGE
# CPU only. No further TEST model forward.
import pandas as pd, numpy as np, math, shutil, json
from scipy.stats import t as student_t

rows=[]
for r in PER_SEED_RESULTS:
    rows.append({
        "seed":int(r["seed"]),
        "core5_accuracy":float(r["gated"]["core5"]["accuracy"]),
        "core5_macro_f1":float(r["gated"]["core5"]["macro_f1"]),
        "gated_target_macro_f1":float(r["gated"]["target_calibrated"]["macro_f1"]),
        "gated_target_map":float(r["gated"]["target_calibrated"]["map"]),
        "facetgate_target_macro_f1":float(r["facetgate"]["target_calibrated"]["macro_f1"]),
        "facetgate_target_map":float(r["facetgate"]["target_calibrated"]["map"]),
        "facetgate_target_exact_set_accuracy":float(r["facetgate"]["target_calibrated"]["exact_set_accuracy"]),
        "facetgate_target_micro_f1":float(r["facetgate"]["target_calibrated"]["micro_f1"]),
        "facetgate_target_samples_f1":float(r["facetgate"]["target_calibrated"]["samples_f1"]),
        "facetgate_target_hamming_loss":float(r["facetgate"]["target_calibrated"]["hamming_loss"]),
        "facetgate_rare_target_macro_f1":float(r["facetgate"]["target_calibrated"]["rare_target_macro_f1"]),
        "target_macro_f1_delta_pp":float(r["facetgate"]["target_macro_f1_delta_pp"]),
        "target_map_delta":float(r["facetgate"]["target_map_delta_vs_gated"]),
        "facetgate_full6_accuracy":float(r["facetgate"]["full6_accuracy"]),
        "facetgate_full6_macro_f1":float(r["facetgate"]["full6_macro_f1"]),
    })

per_seed=pd.DataFrame(rows).sort_values("seed").reset_index(drop=True)
if per_seed.seed.tolist()!=sorted(PUBLICATION_SEEDS):
    raise RuntimeError("Five-seed final result identity drift")

def summarize_metric(values):
    x=np.asarray(values,dtype=np.float64)
    n=len(x)
    mean=float(x.mean())
    sd=float(x.std(ddof=1))
    crit=float(student_t.ppf(0.975,n-1))
    half=crit*sd/math.sqrt(n)
    return {
        "n":n,
        "mean":mean,
        "sample_sd":sd,
        "ci95_low":mean-half,
        "ci95_high":mean+half,
        "min":float(x.min()),
        "max":float(x.max()),
    }

summary=[]
for col in per_seed.columns:
    if col=="seed":
        continue
    z=summarize_metric(per_seed[col].to_numpy())
    summary.append({"metric":col,**z})
summary_df=pd.DataFrame(summary)

final_dir=WORK_ROOT/"FINAL_5SEED_RESULTS"
final_dir.mkdir(parents=True,exist_ok=True)
per_seed.to_csv(final_dir/"qwen3_facetgate_locked_test_per_seed.csv",index=False)
summary_df.to_csv(final_dir/"qwen3_facetgate_locked_test_mean_sd_ci95.csv",index=False)

# Copy all frozen seed ledgers/metrics/manifests into the final package.
for seed in PUBLICATION_SEEDS:
    src=WORK_ROOT/f"seed_{seed}"
    dst=final_dir/f"seed_{seed}"
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True)
    for name in [
        "locked_test_metrics.json",
        "locked_test_prediction_ledger.parquet",
        "locked_test_prediction_ledger.csv",
        "LOCKED_TEST_ONCE_MARKER.json",
        "locked_test_worker_config.json",
    ]:
        p=src/name
        if p.is_file():
            shutil.copy2(p,dst/name)

summary_payload={
    "schema":"M3Offend_Qwen3VL8B_FacetGate_v1_0_5Seed_FINAL_LOCKED_TEST_SUMMARY",
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "facetgate_protocol_sha256":FACETGATE_COMMON_SHA256,
    "gated_protocol_sha256":PARENT_GATED_PROTOCOL_SHA256,
    "qwen_feature_binding_sha256":QWEN_PARENT_INTERFACE_SHA256,
    "test_rows_per_seed":932,
    "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
    "seeds":PUBLICATION_SEEDS,
    "statistics":"mean + sample SD (ddof=1) + two-sided 95% Student-t CI across five predefined seeds",
    "primary_endpoint":"validation-frozen FacetGate Target thresholds on the single locked TEST pass",
    "core5_policy":"untouched matched Gated predictions/thresholds; no Core5 retraining or recalibration",
    "target_policy":"FacetGate replaces only Target; selected validation checkpoint and seven validation thresholds are frozen",
    "per_seed":rows,
    "summary":{r["metric"]:{k:r[k] for k in ["n","mean","sample_sd","ci95_low","ci95_high","min","max"]} for r in summary},
    "no_test_tuning":True,
    "no_selective_rerun":True,
}
atomic_json(final_dir/"qwen3_facetgate_5seed_final_locked_test_summary.json",summary_payload)

manifest={
    "schema":"M3Offend_Qwen3VL8B_FacetGate_v1_0_FINAL_PACKAGE_MANIFEST",
    "pretest_asset_freeze_sha256":sha256_file(WORK_ROOT/"FINAL_PRETEST_ASSET_FREEZE.json"),
    "five_seed_development_gate_sha256":sha256_file(WORK_ROOT/"FIVE_SEED_DEVELOPMENT_GATE_PASS.json"),
    "files":{},
}
for p in sorted(final_dir.rglob("*")):
    if p.is_file() and p.name!="FINAL_PACKAGE_MANIFEST.json":
        manifest["files"][p.relative_to(final_dir).as_posix()]=sha256_file(p)
atomic_json(final_dir/"FINAL_PACKAGE_MANIFEST.json",manifest)

zip_base=WORK_ROOT/"M3Offend_9318_Qwen3VL8B_FacetGate_5Seed_FINAL_LOCKED_TEST_v1_0"
zip_path=Path(shutil.make_archive(str(zip_base),"zip",root_dir=final_dir))

print("\nPRIMARY FINAL TEST — FIVE-SEED SUMMARY")
display(summary_df)
print("\nPER-SEED FINAL TEST")
display(per_seed)
print("\nFinal ZIP:",zip_path)
print("Final ZIP SHA256:",sha256_file(zip_path))
